<a href="https://colab.research.google.com/github/GarretMaloney/DamageAssessment/blob/main/Maloney_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Satellite-Based Remote Sensing Damage Assessment of Five Urban Areas in Jamaica

Hurricane Melissa struck Jamaica on October 28, 2025, as one of the strongest Category 5 hurricanes on record, causing extensive damage across the island with the most severe impacts concentrated in the west. This project applies remote sensing techniques to quantify different forms of damage—structural, vegetation, debris, and flooding—across five urban areas. I approached the project planning to use AI to aid in coding, which enabled me to progress significantly faster than working independently. The scope and intention of the project evolved as I encountered challenges and integrated changes suggested by AI. This report discusses the project scope, methods, successes and shortcomings, and my experience using AI as an assistant in coding and development.

---

## Scope Development

Initially, I planned to perform a building damage assessment mimicking the Unit 6 project and quantify vegetation damage through NDVI analysis. During development, I realized that building damage can occur without structural collapse—flooding and wind can ruin foundations or blow out windows while leaving buildings standing. AI suggested using backscatter as a means of determining flooded areas (lower backscatter) or debris-strewn areas (higher backscatter), which provided additional metrics for quantifying damage remotely. A combined damage index was also created to aid in assessing severity.

---

## Methods

### Area of Interest

Five areas were investigated: Kingston, Negril, Montego Bay, Falmouth, and Ocho Rios. These were selected to capture regional variation in damage. News reports indicated that Montego Bay and Falmouth experienced the most severe impacts, while other areas sustained damage to lesser degrees. Polygons were drawn around each area that approximated the urban extent visible in Google Maps satellite imagery, then clipped to Jamaica's administrative boundary to define clean coastlines and exclude ocean areas from analysis.

### Building Damage

Microsoft Building Footprints were imported from Google Earth Engine and filtered to each area of interest. Per-building damage assessment used SAR coherence calculated from pre- and post-hurricane Sentinel-1 imagery. Buildings were flagged as damaged if mean coherence fell below 0.7 (indicating roof damage or structural issues) or if any pixel within the building showed coherence below 0.5 (severe damage).

### Backscatter Change

Backscatter change analysis identified flooding and wind damage signatures. Pixels with backscatter decreases exceeding 2 dB were classified as flooded, with severity scaled by the magnitude of decrease. Conversely, backscatter increases exceeding 2 dB indicated debris accumulation from wind damage, with severity similarly scaled by magnitude.

### NDVI

Vegetation damage assessment used NDVI change calculated from Sentinel-2 optical imagery. Unlike the other indicators which rely on Sentinel-1 SAR data, this optical-based approach provides an independent data stream, reducing single-sensor bias. Pixels with NDVI decreases exceeding 0.15 were classified as experiencing significant vegetation loss.

### Combined Damage Assessment

A combined damage index synthesized all four indicators at the pixel level:

- **Maximum damage extent**: Pixels flagged by at least one indicator (useful for survey planning)
- **High-confidence damage zones**: Pixels flagged by two or more indicators (priority areas for response)
- **Continuous severity layer**: Generated with the assumption that pixels showing multiple damage signatures indicate more severe impacts

---

## Shortcomings

### Polygon Definition

Polygon definition significantly impacts damage assessments. Including less-developed areas with extensive vegetation led to inflated vegetation damage statistics, which can skew overall damage rankings. Future work should use standardized urban boundaries (e.g., from Global Human Settlement Layer) to ensure consistent area definitions.

### Temporal Resolution

Temporal resolution posed challenges, particularly for optical imagery where cloud cover limited availability. The ephemeral nature of flooding means that delays between hurricane landfall and post-event imaging can miss significant impacts—standing water recedes within days, though residual damage signatures remain detectable through backscatter and NDVI changes. While NDVI and backscatter changes can detect residual damage signatures, these metrics serve as proxies that require careful threshold calibration to accurately quantify impacts.

### Single-Sensor Dependency

Three of the four damage indicators rely on Sentinel-1 SAR data, which introduces single-sensor dependency. While these indicators measure different physical phenomena (coherence vs. backscatter magnitude and direction), issues with Sentinel-1 calibration or processing could systematically bias results.

### Threshold and Weight Validation

Establishing statistically rigorous damage thresholds should be the focus of future work on this project, if possible. Current thresholds were iteratively refined through AI-assisted analysis and reasoning about physical implications, but lack validation against ground-truth damage assessments. Calibrating thresholds and severity weights using field surveys from past hurricanes would substantially improve the reliability and interpretability of results.

The damage weighting scheme similarly lacks statistical validation. The relative importance of structural versus vegetation damage in determining overall severity requires domain expertise and empirical validation. Future work should incorporate ground-truth damage assessments from historical hurricanes to calibrate both thresholds and weights, establishing a validated framework for operational damage assessment.

---

## Reflections on AI-Assisted Development

AI is a powerful tool to iterate changes and write code efficiently but requires critical oversight. A thorough understanding of the data, methods, and project scope is essential for recognizing when AI suggestions diverge from objectives.

Conversely, AI frequently proposed valuable extensions beyond the initial scope, such as the backscatter-based water and wind damage indicators, which enhanced the analysis.

### A Key Learning Experience

A particularly instructive challenge arose when building damage coverage for Ocho Rios appeared incomplete—only a small sliver showed results despite the NDVI layer covering the full area. Initially, I prompted the AI to fix the coherence calculations, but it repeatedly attempted solutions that failed to address the root cause.

After two class periods, I shifted approach and asked the AI to diagnose *why* the coherence layer was incomplete. The diagnostic code revealed that pre- and post-event Sentinel-1 images overlapped by only 6%. This led to developing an automated image pair selection system that balances temporal proximity to the hurricane with sufficient spatial overlap, ensuring both temporal correlation and complete coverage.

### Lessons Learned

This experience highlighted an important lesson: **when encountering problems, request diagnostic analysis and explanations before attempting fixes.** Understanding the root cause enables targeted solutions, whereas immediate attempts at correction may address symptoms rather than underlying issues.

AI lacks comprehensive project context and domain understanding. The most effective approach is to prompt for information and diagnostic tools that support informed human decision-making, rather than delegating decisions to the AI itself. This human-in-the-loop approach leverages AI's computational capabilities while maintaining critical oversight and domain expertise.

---

# Hurricane Melissa Damage Assessment - Jamaica

**Multi-Indicator Satellite-Based Damage Detection**

This notebook performs comprehensive hurricane damage assessment using satellite remote sensing data from Hurricane Melissa (Category 5), which impacted Jamaica on October 28, 2025.

## Methodology Overview

We combine three complementary satellite-based damage indicators:

1. **Sentinel-1 SAR Coherence** - Detects structural collapse and debris
2. **Sentinel-1 Backscatter Change** - Identifies flooding and surface roughness changes
3. **Sentinel-2 NDVI Change** - Measures vegetation loss and environmental damage

## Study Areas

Five regions across Jamaica with varying reported impact levels:
- **Kingston** (Southeast) - Indirect impact
- **Ocho Rios** (North-central) - Minimal impact
- **Negril** (West) - Moderate impact
- **Montego Bay** (Northwest) - Severe impact
- **Falmouth** (North-central coast) - Reportedly "all but destroyed"

In [ ]:
# ============================================================================
# SECTION 1: SETUP AND AUTHENTICATION
# ============================================================================

import ee
import geemap
import shapely.wkt
from datetime import datetime
import numpy as np

ee.Authenticate()
ee.Initialize(project="maloney-geog-473")  # Replace with your project ID

print("✓ Google Earth Engine initialized successfully")

✓ Google Earth Engine initialized successfully


## Section 2: Define Areas of Interest (AOIs)

We define five study regions across Jamaica using polygon geometries. Each AOI:
- Roughly mimics the extent from google maps
- Clipped to Jamaica's land boundary to exclude ocean areas
- Associated with building footprints from the Microsoft Buildings dataset to calculate building damage

In [ ]:
# ============================================================================
# SECTION 2: DEFINE AREAS OF INTEREST
# ============================================================================

# Load Jamaica boundary and building footprints
jamaica = ee.FeatureCollection("FAO/GAUL/2015/level0") \
    .filter(ee.Filter.eq('ADM0_NAME', 'Jamaica'))
jamaica_buildings = ee.FeatureCollection("projects/sat-io/open-datasets/MSBuildings/Jamaica")

# Define custom polygon coordinates for each region
# Coordinates in [longitude, latitude] format for Earth Engine
aoi_definitions = {
    'Kingston': [
        [-76.855177, 17.956113],
        [-76.871921, 18.058523],
        [-76.791081, 18.079647],
        [-76.729280, 18.035039],
        [-76.728544, 17.956788],
        [-76.855177, 17.956113]
    ],
    'Ocho Rios': [
        [-77.099180, 18.387197],
        [-77.117265, 18.416621],
        [-77.063238, 18.411412],
        [-77.047445, 18.390889],
        [-77.099180, 18.387197]
    ],
    'Negril': [
        [-78.338644, 18.281032],
        [-78.337270, 18.254135],
        [-78.360702, 18.271415],
        [-78.358900, 18.288285],
        [-78.338644, 18.281032]
    ],
    'Montego Bay': [
        [-77.858437, 18.462327],
        [-77.877390, 18.427820],
        [-77.916657, 18.432138],
        [-77.996527, 18.510196],
        [-77.889953, 18.517332],
        [-77.858437, 18.462327]
    ],
    'Falmouth': [
        [-77.664838, 18.482109],
        [-77.674730, 18.489813],
        [-77.646921, 18.505197],
        [-77.648938, 18.488511],
        [-77.664838, 18.482109]
    ]
}

# Create AOIs and building collections
aois = {}
buildings = {}

print("Creating custom area of interest boundaries...")
print("="*70)

for name, coords in aoi_definitions.items():
    # Create polygon and clip to Jamaica boundary for clean shoreline
    polygon = ee.Geometry.Polygon(coords)
    aois[name] = polygon.intersection(jamaica.geometry(), maxError=10)
    buildings[name] = jamaica_buildings.filterBounds(aois[name])

    # Calculate statistics
    area_km2 = aois[name].area().getInfo() / 1e6
    num_buildings = buildings[name].size().getInfo()

    print(f"✓ {name:15s}: {area_km2:6.1f} km² | {num_buildings:,} buildings")

print("="*70)
print("\nCustom polygons clipped to Jamaica administrative boundary")
print("Clean shorelines with no ocean areas included in analysis")
print("="*70)

Creating custom area of interest boundaries...
✓ Kingston       :  151.9 km² | 108,235 buildings
✓ Ocho Rios      :   14.2 km² | 7,909 buildings
✓ Negril         :    4.6 km² | 2,954 buildings
✓ Montego Bay    :   54.0 km² | 32,914 buildings
✓ Falmouth       :    2.8 km² | 2,103 buildings

Custom polygons clipped to Jamaica administrative boundary
Clean shorelines with no ocean areas included in analysis


## Section 3: Helper Functions

### Image Selection Strategy

For accurate damage detection, we need to carefully select pre and post-hurricane images. The optimal image pair must balance two competing factors:

1. **Temporal Baseline**: Time between images (shorter is better, ideally 7-14 days)
2. **Coverage Overlap**: Both images must cover >80% of the AOI

**Why this matters**: Adjacent satellite swaths from the same pass can have minimal overlap, and longer time gaps introduce natural surface changes (temporal decorrelation) that mask actual damage.

In [ ]:

# ============================================================================
# SECTION 3: HELPER FUNCTIONS
# ============================================================================

def get_s1_candidates(aoi, start_date, end_date):
    """Query Sentinel-1 images and return metadata with pixel coverage."""
    s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
            .filterBounds(aoi)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .select("VV"))

    count = s1.size().getInfo()
    img_list = s1.sort("system:time_start").toList(count)

    candidates = []
    for i in range(count):
        img = ee.Image(img_list.get(i))
        img_info = img.getInfo()
        img_date = datetime.fromtimestamp(
            img_info['properties']['system:time_start']/1000
        ).strftime('%Y-%m-%d %H:%M')

        pixels = img.clip(aoi).reduceRegion(
            reducer=ee.Reducer.count(),
            geometry=aoi,
            scale=10,
            maxPixels=1e9
        ).getInfo().get('VV', 0)

        candidates.append({
            'date': img_date,
            'id': img_info['id'],
            'pixels': pixels,
            'image': img
        })

    return candidates


def find_optimal_image_pair(aoi, area_name, hurricane_date, pre_candidates, post_candidates):
    """Find optimal pre/post pair balancing coverage and temporal baseline."""
    print(f"\n{'='*70}")
    print(f"FINDING OPTIMAL IMAGE PAIR: {area_name}")
    print(f"{'='*70}")

    hurricane_dt = datetime.strptime(hurricane_date, '%Y-%m-%d')
    max_pixels = max([c['pixels'] for c in pre_candidates + post_candidates])
    coverage_threshold = 0.8 * max_pixels

    good_pre = [c for c in pre_candidates if c['pixels'] > coverage_threshold]
    good_post = [c for c in post_candidates if c['pixels'] > coverage_threshold]

    print(f"Images with >80% coverage: Pre={len(good_pre)}, Post={len(good_post)}")

    if not good_pre or not good_post:
        good_pre = sorted(pre_candidates, key=lambda x: x['pixels'], reverse=True)[:3]
        good_post = sorted(post_candidates, key=lambda x: x['pixels'], reverse=True)[:3]

    best_pair = None
    best_baseline = float('inf')

    for pre in good_pre:
        pre_dt = datetime.strptime(pre['date'], '%Y-%m-%d %H:%M')
        days_before = (hurricane_dt - pre_dt).days
        if days_before < 0 or days_before > 20:
            continue

        for post in good_post:
            post_dt = datetime.strptime(post['date'], '%Y-%m-%d %H:%M')
            days_after = (post_dt - hurricane_dt).days
            if days_after < 0 or days_after > 14:
                continue

            baseline = (post_dt - pre_dt).days
            if baseline < best_baseline and baseline <= 21:
                best_baseline = baseline
                best_pair = (pre, post, days_before, days_after)

    if not best_pair:
        best_pre = sorted(good_pre, key=lambda x: x['pixels'], reverse=True)[0]
        best_post = sorted(good_post, key=lambda x: x['pixels'], reverse=True)[0]
        pre_dt = datetime.strptime(best_pre['date'], '%Y-%m-%d %H:%M')
        post_dt = datetime.strptime(best_post['date'], '%Y-%m-%d %H:%M')
        days_before = (hurricane_dt - pre_dt).days
        days_after = (post_dt - hurricane_dt).days
        best_baseline = (post_dt - pre_dt).days
        best_pair = (best_pre, best_post, days_before, days_after)

    best_pre, best_post, days_before, days_after = best_pair
    overlap = min(best_pre['pixels'], best_post['pixels'])
    overlap_pct = (overlap / max(best_pre['pixels'], best_post['pixels'])) * 100

    print(f"\n✓ OPTIMAL PAIR:")
    print(f"  Pre:  {best_pre['date']} ({days_before}d before) | {best_pre['pixels']:,} px")
    print(f"  Post: {best_post['date']} ({days_after}d after) | {best_post['pixels']:,} px")
    print(f"  Temporal baseline: {best_baseline} days | Overlap: {overlap_pct:.1f}%")
    print(f"{'='*70}\n")

    return best_pre['image'], best_post['image']


def mask_s2_clouds(image):
    """Mask clouds and cirrus in Sentinel-2 imagery using QA60 band."""
    qa = image.select("QA60")
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask).divide(10000)

print("✓ Helper functions defined")

✓ Helper functions defined


# Section 4: Comprehensive Damage Assessment Function

## Multi-Indicator Approach

We use four complementary indicators to capture different types of hurricane damage:

---

### 1. SAR Coherence (Structural Damage)

**What it measures**: Surface change between pre- and post-hurricane SAR images  

**Formula**: `coherence = 1 - |pre - post| / (pre + post)`  

**Interpretation**:
- **High coherence (0.7-1.0)** = Surface unchanged = No damage
- **Medium coherence (0.5-0.7)** = Minor surface change = Possible damage
- **Low coherence (0-0.5)** = Major surface change = Structural damage

**Detection threshold**: Buildings with mean coherence < 0.5 OR any pixel < 0.3 are flagged as damaged

**Per-Building Assessment**: Each building footprint is assessed individually rather than pixel-by-pixel. A building is considered damaged if:
- Average coherence across the building < 0.7 (moderate structural issues), OR
- Any pixel within the building < 0.3 (severe damage detected)

**Limitations**:
- Misses flooding where water remains smooth
- Cannot detect roof damage without collapse
- Does not capture interior damage

---

### 2. Backscatter Intensity Change (Water & Wind Damage)

**What it measures**: Change in radar return signal strength measured in decibels (dB)

**Formula**: `dB change = post_dB - pre_dB`

**Interpretation**:

#### Water Damage (Flooding)
- **Decrease < -2 dB** = Flooding or standing water
- More negative = More severe flooding
- **Why**: Smooth water reflects radar away from sensor → lower return signal

#### Wind Damage (Debris)
- **Increase > +2 dB** = Debris accumulation or material displacement  
- More positive = More severe debris field
- **Why**: Rough debris scatters radar back to sensor → higher return signal

**Continuous Severity Scale**:
- Water: 0 dB (no damage) → 8+ dB decrease (catastrophic flooding)
- Wind: 0 dB (no damage) → 8+ dB increase (severe debris)

**Limitations**:
- Ambiguous in some cases - decrease could be flooding OR building collapse
- Affected by soil moisture changes
- May miss damage that doesn't change surface roughness

---

### 3. NDVI Change (Vegetation & Environmental Damage)

**What it measures**: Normalized Difference Vegetation Index from optical satellite imagery

**Formula**: `NDVI = (NIR - Red) / (NIR + Red)`

**Interpretation**:
- **Decrease > 0.15** = Significant vegetation loss
- Typical range: -0.3 to -0.6 for severe defoliation/uprooting

**Continuous Severity Scale**:
- 0 (no change) → 0.6+ (catastrophic vegetation loss)

**Limitations**:
- **Cloud cover** limits availability (common after hurricanes)
- Does not detect structural damage to buildings
- Can be affected by flooding (appears as vegetation loss)

---

### 4. Combined Damage Index

**Purpose**: Synthesizes all indicators to provide a holistic damage assessment

**Three Metrics Calculated**:

#### a) Any Indicator (Maximum Extent)
- **Formula**: `Structural OR Water OR Wind OR Vegetation`
- **What it shows**: Percentage of pixels flagged by **at least one** indicator
- **Use case**: Identifies the full extent of the impact zone
- **Interpretation**: Broad, sensitive measure - captures all potentially affected areas

#### b) High Confidence Damage (Validated Core)
- **Formula**: Count of indicators ≥ 2 per pixel
- **What it shows**: Percentage of pixels flagged by **two or more** indicators
- **Use case**: Filters false positives, identifies confirmed damage
- **Interpretation**: Conservative, reliable measure - priority areas for response

#### c) Continuous Composite Score (Severity Gradient)
- **Formula**: `0.25 × Structural + 0.25 × Water + 0.25 × Wind + 0.25 × Vegetation`
- **Scale**: 0.0 (no damage) to 1.0 (maximum damage)
- **What it shows**: Weighted composite severity for each pixel
- **Use case**: Visualization and prioritization mapping
- **Color mapping**:
  - White (0.0) = No damage
  - Yellow (0.25) = Single indicator, minor damage
  - Orange (0.5) = Multiple indicators, moderate damage
  - Red (1.0) = All indicators, catastrophic damage

**Interpretation Guide**:
- **Any Indicator = 45%** → 45% of area shows some type of damage (extent)
- **High Confidence = 18%** → 18% has reliable, multi-indicator confirmed damage (core)
- **Continuous map** → Shows gradient from minor (yellow) to severe (red) damage

**Why all three?**: Each serves a different purpose - extent (planning), confidence (validation), and severity (prioritization)

---

## Point-Based Scoring System

Each indicator is scored on a 0-10 point scale:

- **Structural (Buildings)**: Based on % of buildings damaged (not pixel area)
  - 0 buildings damaged = 0 points
  - 100% buildings damaged = 10 points

- **Water Damage**: Extent (0-5 pts) + Intensity (0-5 pts) = 0-10 total
  - Extent: Based on % of area with flooding signature
  - Intensity: Based on average severity in flooded areas

- **Wind Damage**: Extent (0-5 pts) + Intensity (0-5 pts) = 0-10 total
  - Extent: Based on % of area with debris signature  
  - Intensity: Based on average severity in debris-affected areas

- **Vegetation Loss**: Based on % of area # Section 4: Damage Assessment Methodology

## Overview

Four complementary satellite-based indicators capture different hurricane damage types. Each generates **two outputs**: (1) binary masks for statistics, (2) continuous severity layers for mapping.

---

## Indicator 1: Structural Damage (SAR Coherence)

**Measures**: Surface change between pre/post-hurricane radar images  
**Formula**: `coherence = 1 - |pre - post| / (pre + post)`  
**Range**: 0.0 (changed) to 1.0 (unchanged)

### Thresholds
- **< 0.5**: Structural damage detected
- **< 0.3**: Severe damage (building collapse likely)

### Per-Building Assessment
Each building footprint assessed individually:
- **Damaged if**: Mean coherence < 0.5 OR any pixel < 0.3
- **Output**: Count and percentage of damaged buildings

### Limitations
- Cannot detect flooding (water is smooth)
- Misses roof damage without collapse
- No interior damage detection

---

## Indicator 2: Water & Wind Damage (Backscatter Change)

**Measures**: Radar signal strength change (decibels)  
**Formula**: `dB change = post - pre`

### Water Damage (Negative Change)
- **Threshold**: < -2 dB = flooding/standing water
- **Why**: Smooth water reflects radar away → weaker signal
- **Severity scale**: 0 to 8+ dB decrease

### Wind Damage (Positive Change)
- **Threshold**: > +2 dB = debris/material displacement
- **Why**: Rough debris scatters radar back → stronger signal
- **Severity scale**: 0 to 8+ dB increase

### Scoring
Both use **Extent** (% area affected) + **Intensity** (average severity)
- Extent: 0-5 points
- Intensity: 0-5 points
- Total: 0-10 points

### Limitations
- Ambiguous signals (decrease could be flood OR collapse)
- Affected by soil moisture
- Misses damage without roughness change

---

## Indicator 3: Vegetation Loss (NDVI Change)

**Measures**: Vegetation health from optical imagery  
**Formula**: `NDVI = (NIR - Red) / (NIR + Red)`  
**Range**: -1.0 to +1.0

### Thresholds
- **< -0.15**: Significant vegetation loss (defoliation/uprooting)
- **Typical damage**: -0.3 to -0.6

### Outputs
- **Mean NDVI change**: Average across entire area (includes undamaged pixels)
- **Vegetation loss extent**: % of area exceeding threshold
- **Continuous layer**: Pixel-by-pixel severity (0 to 0.6+ scale)

### Scoring
Based on extent only: 0-10 points

### Limitations
- **Cloud dependent**: Common issue post-hurricane
- No structural damage detection
- Flooding appears as vegetation loss

---

## Indicator 4: Combined Damage Index

Synthesizes all indicators into three complementary metrics:

### a) Any Indicator (Maximum Extent)
**What**: Pixels flagged by ≥1 indicator  
**Use**: Defines total impact zone for survey planning  
**Output**: "45% of area shows some damage"

### b) High Confidence (Validated Core)
**What**: Pixels flagged by ≥2 indicators  
**Use**: Confirmed damage zones for priority response  
**Output**: "18% of area has reliable multi-indicator damage"

### c) Continuous Composite (Severity Map)
**What**: Weighted average of all severity layers  
**Formula**: `0.25 × (Structural + Water + Wind + Vegetation)`  
**Range**: 0.0 (no damage) to 1.0 (catastrophic)  
**Use**: Visualization and resource prioritization

**Note**: This continuous layer is used for **mapping only**. The statistics you see in the output (Any Indicator %, High Confidence %) come from the binary masks, not the continuous layer.

### Interpretation
- **Any = 45%**: Full extent - what area needs survey
- **High confidence = 18%**: Confirmed core - where to focus first
- **Continuous map**: White→Yellow→Orange→Red shows severity gradient

---

## Scoring System

Total: 40 points (4 indicators × 10 points each)

| Indicator | Scoring Method |
|-----------|----------------|
| **Structural** | Buildings damaged: 0% = 0 pts, 100% = 10 pts |
| **Water** | Extent (0-5) + Intensity (0-5) |
| **Wind** | Extent (0-5) + Intensity (0-5) |
| **Vegetation** | Extent only: 0% = 0 pts, 100% = 10 pts |

Final score normalized to 0-100% severity scale.

---

## Data & Timeline

**Sources**: Sentinel-1 (radar), Sentinel-2 (optical), MS Building Footprints  
**Resolution**: 10m pixels  
**Hurricane Melissa**: October 28, 2025  
**Pre-event**: Oct 1-27 (optimal pair selected)  
**Post-event**: Oct 29 - Nov 10 (optimal pair selected)  
**Temporal baseline**: ≤21 days preferred

---with significant NDVI decrease
  - 0% vegetation loss = 0 points
  - 100% vegetation loss = 10 points

**Total Possible**: 40 points (converted to 0-100% severity scale for final scoring)

---

## Data Sources

- **SAR**: Sentinel-1 C-band radar (10m resolution, all-weather)
- **Optical**: Sentinel-2 multispectral imagery (10m resolution, cloud-dependent)
- **Buildings**: Microsoft Building Footprints dataset
- **Boundaries**: Custom polygons clipped to Jamaica administrative boundary

---

## Temporal Baseline

- **Pre-event window**: Oct 1-27, 2025 (optimal pair selected algorithmically)
- **Hurricane date**: Oct 28, 2025
- **Post-event window**: Oct 29 - Nov 10, 2025 (optimal pair selected algorithmically)
- **Target temporal baseline**: ≤21 days between pre/post images for valid coherence

---

In [ ]:
# ============================================================================
# SECTION 4: COMPREHENSIVE DAMAGE ASSESSMENT FUNCTION
# ============================================================================

def assess_building_damage(buildings, coherence_image, aoi, area_name):
    """
    Per-building damage assessment using zonal statistics.
    A building is damaged if mean coherence < 0.5 OR min coherence < 0.3
    """

    print(f"   Assessing individual buildings...")

    total_buildings = buildings.size().getInfo()

    if total_buildings == 0:
        return 0, 0, None

    # Calculate mean and min coherence for each building
    stats = coherence_image.reduceRegions(
        collection=buildings,
        reducer=ee.Reducer.mean().combine(ee.Reducer.min(), '', True),
        scale=10
    )

    # Filter for damaged buildings
    damaged = stats.filter(
        ee.Filter.Or(
            ee.Filter.lt('mean', 0.7),
            ee.Filter.lt('min', 0.3)
        )
    )

    damaged_count = damaged.size().getInfo()
    damage_pct = (damaged_count / total_buildings) * 100

    print(f"   Buildings assessed: {total_buildings:,}")
    print(f"   Buildings damaged: {damaged_count:,} ({damage_pct:.1f}%)")

    return damage_pct, damaged_count, damaged


def comprehensive_damage_assessment(aoi, buildings, area_name,
                                   pre_image_s1, post_image_s1,
                                   pre_event_dates, post_event_dates):
    """Multi-indicator damage assessment combining SAR and optical data."""

    print(f"\n{'='*70}")
    print(f"COMPREHENSIVE DAMAGE ASSESSMENT: {area_name}")
    print(f"{'='*70}\n")

    # INDICATOR 1: SAR COHERENCE (STRUCTURAL COLLAPSE)
    print("1. SAR Coherence (structural collapse)...")

    pre_linear = ee.Image(10).pow(pre_image_s1.divide(10))
    post_linear = ee.Image(10).pow(post_image_s1.divide(10))

    numerator = pre_linear.subtract(post_linear).abs()
    denominator = pre_linear.add(post_linear)
    coherence = numerator.divide(denominator).multiply(-1).add(1).rename("coherence").clip(aoi)

    structural_damage = coherence.lt(0.5).rename("structural")

    coh_mean = coherence.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=10, maxPixels=1e9
    ).getInfo().get('coherence', 0)

    print(f"   Mean coherence: {coh_mean:.3f}")

    # Per-building assessment
    building_damage_pct, damaged_count, assessed_buildings = assess_building_damage(
        buildings, coherence, aoi, area_name
    )

    # INDICATOR 2: BACKSCATTER CHANGE - WATER & WIND DAMAGE
    print("\n2. Backscatter Change (water & wind damage)...")

    backscatter_change = post_image_s1.subtract(pre_image_s1).rename("db_change").clip(aoi)

    db_stats = backscatter_change.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
        geometry=aoi, scale=10, maxPixels=1e9
    ).getInfo()

    print(f"   Mean dB change: {db_stats.get('db_change_mean', 0):.2f} dB")

    # WATER DAMAGE - Continuous layer
    water_damage_continuous = backscatter_change.multiply(-1).clamp(0, 10).rename("water_continuous")

    # WIND DAMAGE - Continuous layer
    wind_damage_continuous = backscatter_change.clamp(0, 10).rename("wind_continuous")

    # Binary masks
    water_damage_binary = backscatter_change.lt(-2).rename("water")
    wind_damage_binary = backscatter_change.gt(2).rename("wind")

    water_pct = water_damage_binary.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=10, maxPixels=1e9
    ).getInfo().get('water', 0) * 100

    wind_pct = wind_damage_binary.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=10, maxPixels=1e9
    ).getInfo().get('wind', 0) * 100

    print(f"   Water damage extent: {water_pct:.1f}%")
    print(f"   Wind damage extent: {wind_pct:.1f}%")

    # Calculate severity for point scoring
    water_severity = backscatter_change.where(
        backscatter_change.gte(-2), 0
    ).multiply(-1).divide(2).clamp(0, 5)

    water_severity_weighted = water_severity.updateMask(water_damage_binary)

    water_intensity = water_severity_weighted.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=10, maxPixels=1e9
    ).getInfo().get('db_change', 0)

    water_extent_pts = min((water_pct / 100) * 5, 5)
    water_intensity_pts = min(water_intensity, 5)
    water_points = water_extent_pts + water_intensity_pts

    print(f"   Water damage (points): {water_points:.2f}/10 (extent: {water_extent_pts:.2f}, intensity: {water_intensity_pts:.2f})")

    wind_severity = backscatter_change.where(
        backscatter_change.lte(2), 0
    ).divide(2).clamp(0, 5)

    wind_severity_weighted = wind_severity.updateMask(wind_damage_binary)

    wind_intensity = wind_severity_weighted.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=10, maxPixels=1e9
    ).getInfo().get('db_change', 0)

    wind_extent_pts = min((wind_pct / 100) * 5, 5)
    wind_intensity_pts = min(wind_intensity, 5)
    wind_points = wind_extent_pts + wind_intensity_pts

    print(f"   Wind damage (points): {wind_points:.2f}/10 (extent: {wind_extent_pts:.2f}, intensity: {wind_intensity_pts:.2f})")

    # INDICATOR 3: NDVI CHANGE (VEGETATION LOSS)
    print("\n3. Vegetation Loss (defoliation/uprooting)...")

    s2_pre = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterBounds(aoi)
                .filterDate(pre_event_dates[0], pre_event_dates[1])
                .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
                .map(mask_s2_clouds).median().clip(aoi))

    s2_post = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                 .filterBounds(aoi)
                 .filterDate(post_event_dates[0], post_event_dates[1])
                 .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
                 .map(mask_s2_clouds).median().clip(aoi))

    ndvi_pre = s2_pre.normalizedDifference(["B8", "B4"]).rename("NDVI_pre")
    ndvi_post = s2_post.normalizedDifference(["B8", "B4"]).rename("NDVI_post")
    ndvi_change = ndvi_post.subtract(ndvi_pre).rename("NDVI_change")

    # VEGETATION DAMAGE - Continuous layer
    vegetation_damage_continuous = ndvi_change.multiply(-1).clamp(0, 1).rename("vegetation_continuous")

    # Binary mask
    veg_damage_binary = ndvi_change.lt(-0.15).rename("vegetation")

    ndvi_mean = ndvi_change.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=10, maxPixels=1e9
    ).getInfo().get('NDVI_change', 0)

    veg_pct = veg_damage_binary.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=10, maxPixels=1e9
    ).getInfo().get('vegetation', 0) * 100

    print(f"   Mean NDVI change: {ndvi_mean:.3f} (entire area)")
    print(f"   Vegetation loss extent: {veg_pct:.1f}%")

    # Convert to points
    structural_points = min((building_damage_pct / 100) * 10, 10)
    vegetation_points = min((veg_pct / 100) * 10, 10)

    # COMBINED METRICS
    print("\n4. Combined Damage Index...")

    any_damage = structural_damage.Or(water_damage_binary).Or(wind_damage_binary).Or(veg_damage_binary).rename("any_damage")
    damage_count = structural_damage.add(water_damage_binary).add(wind_damage_binary).add(veg_damage_binary)
    high_confidence = damage_count.gte(2).rename("high_confidence")

    # COMBINED DAMAGE - Continuous layer
    combined_damage_continuous = (
        coherence.multiply(-1).add(1).multiply(0.25)
        .add(water_damage_continuous.divide(10).multiply(0.25))
        .add(wind_damage_continuous.divide(10).multiply(0.25))
        .add(vegetation_damage_continuous.multiply(0.25))
    ).rename("combined_continuous")

    any_damage_pct = any_damage.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=10, maxPixels=1e9
    ).getInfo().get('any_damage', 0) * 100

    high_conf_pct = high_confidence.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=10, maxPixels=1e9
    ).getInfo().get('high_confidence', 0) * 100

    print(f"   Any indicator (1+ flags):        {any_damage_pct:.1f}%")
    print(f"   High confidence (2+ flags):      {high_conf_pct:.1f}%")

    # TOTAL DAMAGE SCORE
    total_points = structural_points + water_points + wind_points + vegetation_points
    normalized_score = (total_points / 40) * 100

    print(f"\n{'='*70}")
    print(f"SUMMARY: {area_name}")
    print(f"  DAMAGE SCORES (out of 10 each):")
    print(f"    Structural (Buildings):  {structural_points:.2f}/10  ({damaged_count:,} buildings)")
    print(f"    Water Damage:            {water_points:.2f}/10")
    print(f"    Wind Damage:             {wind_points:.2f}/10")
    print(f"    Vegetation Loss:         {vegetation_points:.2f}/10")
    print(f"  TOTAL: {total_points:.2f}/40 points ({normalized_score:.1f}% severity)")
    print(f"{'='*70}\n")

    return {
        'coherence': coherence,
        'backscatter_change': backscatter_change,
        'ndvi_change': ndvi_change,
        'structural_damage': structural_damage.clip(buildings),
        'water_damage': water_damage_binary,
        'wind_damage': wind_damage_binary,
        'vegetation_damage': veg_damage_binary,
        'water_damage_continuous': water_damage_continuous,
        'wind_damage_continuous': wind_damage_continuous,
        'vegetation_damage_continuous': vegetation_damage_continuous,
        'combined_damage_continuous': combined_damage_continuous,
        'any_damage': any_damage,
        'high_confidence': high_confidence,
        'water_pct': water_pct,
        'wind_pct': wind_pct,
        'vegetation_pct': veg_pct,
        'building_damage_pct': building_damage_pct,
        'damaged_building_count': damaged_count,
        'assessed_buildings': assessed_buildings,
        'structural_points': structural_points,
        'water_points': water_points,
        'wind_points': wind_points,
        'vegetation_points': vegetation_points,
        'total_points': total_points,
        'normalized_score': normalized_score,
        'water_intensity': water_intensity,
        'wind_intensity': wind_intensity
    }

print("✓ Assessment function defined")

✓ Assessment function defined


## Section 5: Process All Regions

Now we'll run the comprehensive damage assessment for all five regions across Jamaica.

### Processing Workflow

For each region, the system automatically:

1. **Searches for Sentinel-1 imagery**
   - Pre-event window: October 1-27, 2025
   - Post-event window: October 29 - November 10, 2025
   - Filters for VV polarization and IW (Interferometric Wide swath) mode

2. **Selects optimal image pairs**
   - Balances temporal baseline (≤21 days preferred)
   - Maximizes spatial coverage (>80% threshold)
   - Minimizes days from hurricane date
   - Ensures overlap between pre/post images

3. **Computes all damage indicators**
   - SAR coherence for structural damage
   - Backscatter change for water/wind damage
   - NDVI change for vegetation loss
   - Per-building damage assessment

4. **Validates and stores results**
   - Error handling ensures one failed region doesn't stop others
   - Progress tracking shows which region is currently processing
   - Results stored in `all_results` dictionary for analysis

### Event Timeline

- **Hurricane Melissa landfall**: October 28, 2025
- **Pre-event baseline**: October 1-27, 2025 (optimal pair selected algorithmically)
- **Post-event assessment**: October 29 - November 10, 2025 (optimal pair selected algorithmically)

### Regions Analyzed

1. **Kingston** - Capital city and metropolitan area
2. **Ocho Rios** - North coast tourist town
3. **Negril** - West coast beach resort area
4. **Montego Bay** - Major city and tourism hub
5. **Falmouth** - Historic port town

### Error Handling

The system includes robust error handling:
- Skips regions with insufficient imagery (validation check)
- Continues processing if one region times out or fails
- Reports failed regions at completion
- Provides diagnostic information on Sentinel-2 coverage (cloud limitations)

### Expected Output

For each successfully processed region, you'll see:
- Number of Sentinel-1 images found
- Optimal image pair dates and temporal baseline
- Mean coherence values
- Building damage counts and percentages
- Water, wind, and vegetation damage metrics
- Combined damage indices
- Point-based severity scores (0-40 scale)



In [ ]:
# ============================================================================
# SECTION 5: PROCESS ALL REGIONS
# ============================================================================

HURRICANE_DATE = "2025-10-28"
pre_event_dates = ("2025-10-01", "2025-10-15")
post_event_dates = ("2025-10-31", "2025-11-17")

all_results = {}

print("="*70)
print("PROCESSING ALL REGIONS")
print("="*70)

# Process each region with error handling and progress tracking
for idx, name in enumerate(aois.keys(), 1):
    print(f"\n{'='*70}")
    print(f"PROCESSING {name.upper()} ({idx}/{len(aois)})")
    print(f"{'='*70}")

    try:
        # Get Sentinel-1 candidatesc
        print(f"Searching for Sentinel-1 images...")
        pre_cands = get_s1_candidates(aois[name], "2025-10-01", "2025-10-27")
        post_cands = get_s1_candidates(aois[name], "2025-10-29", "2025-11-10")

        print(f"Found {len(pre_cands)} pre-event and {len(post_cands)} post-event S1 images")

        if len(pre_cands) == 0 or len(post_cands) == 0:
            print(f"⚠️  WARNING: Insufficient imagery for {name}, skipping...")
            continue

        # Find optimal image pair
        pre_img, post_img = find_optimal_image_pair(
            aois[name], name, HURRICANE_DATE, pre_cands, post_cands
        )

        # Run damage assessment
        results = comprehensive_damage_assessment(
            aois[name], buildings[name], name, pre_img, post_img,
            pre_event_dates, post_event_dates
        )

        all_results[name] = results
        print(f"✓ {name} completed successfully")

    except Exception as e:
        print(f"❌ ERROR processing {name}: {str(e)}")
        print(f"Skipping {name} and continuing with next region...")
        continue

print("\n" + "="*70)
print(f"✓ COMPLETED: {len(all_results)}/{len(aois)} REGIONS PROCESSED")
if len(all_results) < len(aois):
    failed = [name for name in aois.keys() if name not in all_results]
    print(f"⚠️  Failed regions: {', '.join(failed)}")
print("="*70)



PROCESSING ALL REGIONS

PROCESSING KINGSTON (1/5)
Searching for Sentinel-1 images...
Found 6 pre-event and 5 post-event S1 images

FINDING OPTIMAL IMAGE PAIR: Kingston
Images with >80% coverage: Pre=6, Post=5

✓ OPTIMAL PAIR:
  Pre:  2025-10-22 10:56 (5d before) | 1,514,337 px
  Post: 2025-10-29 23:18 (1d after) | 1,514,274 px
  Temporal baseline: 7 days | Overlap: 100.0%


COMPREHENSIVE DAMAGE ASSESSMENT: Kingston

1. SAR Coherence (structural collapse)...
   Mean coherence: 0.614
   Assessing individual buildings...
   Buildings assessed: 108,235
   Buildings damaged: 61,966 (57.3%)

2. Backscatter Change (water & wind damage)...
   Mean dB change: 0.82 dB
   Water damage extent: 27.5%
   Wind damage extent: 38.9%
   Water damage (points): 3.95/10 (extent: 1.37, intensity: 2.58)
   Wind damage (points): 4.68/10 (extent: 1.95, intensity: 2.74)

3. Vegetation Loss (defoliation/uprooting)...
   Mean NDVI change: -0.040 (entire area)
   Vegetation loss extent: 15.8%

4. Combined Damage I

## Section 6: Weighted Damage Scoring

### The Problem with Equal Weighting

Simple averaging (25% per indicator) doesn't adequately capture damage severity.

### Solution: Context-Aware Weighting Schemes

We apply four different weighting approaches:

| Scheme | Weights | Best For |
|--------|---------|----------|
| **Urban/Infrastructure** | 40% structural, 30% debris, 20% flood, 10% veg | Reconstruction planning |
| **Environmental/Total** | 35% veg, 25% structural, 25% flood, 15% debris | Cat 5 hurricanes, total devastation |
| **Adaptive Hurricane** | Varies by damage profile | Context-aware assessment |
| **Severity-Weighted** | Exponential scaling for >60% values | Emphasizing catastrophic damage |

In [ ]:
# ============================================================================
# SECTION 6: WEIGHTED DAMAGE SCORING
# ============================================================================

def calculate_weighted_scores(results_dict, area_name):
    """Calculate damage scores using point-based weighting schemes."""

    print(f"\n{'='*70}")
    print(f"WEIGHTED ANALYSIS: {area_name}")
    print(f"{'='*70}")

    # Extract point scores (using the correct variable names)
    structural_pts = results_dict['structural_points']
    water_pts = results_dict['water_points']
    wind_pts = results_dict['wind_points']
    vegetation_pts = results_dict['vegetation_points']

    print(f"Individual Scores (out of 10):")
    print(f"  Structural Collapse:  {structural_pts:.2f}/10")
    print(f"  Water Damage:         {water_pts:.2f}/10")
    print(f"  Wind Damage:          {wind_pts:.2f}/10")
    print(f"  Vegetation Loss:      {vegetation_pts:.2f}/10")

    # Scheme 1: Urban/Infrastructure Focus
    # Emphasizes structural and wind damage
    urban_score = (structural_pts * 4.0 + wind_pts * 3.0 +
                   water_pts * 2.0 + vegetation_pts * 1.0) / 10.0
    urban_score = (urban_score / 10) * 100  # Convert to 0-100 scale
    print(f"\n1. Urban/Infrastructure Focus: {urban_score:.1f}/100")

    # Scheme 2: Environmental/Total Devastation
    # Balanced across all indicators with emphasis on vegetation
    environmental_score = (vegetation_pts * 3.5 + structural_pts * 2.5 +
                          water_pts * 2.5 + wind_pts * 1.5) / 10.0
    environmental_score = (environmental_score / 10) * 100
    print(f"2. Environmental/Total: {environmental_score:.1f}/100")

    # Scheme 3: Adaptive Profile-Based
    # Adjusts weights based on damage pattern
    if vegetation_pts > 6:
        weights = {'vegetation': 4.0, 'structural': 2.5, 'wind': 2.0, 'water': 1.5}
        profile = "CATASTROPHIC VEGETATION"
    elif structural_pts > 3:
        weights = {'structural': 4.5, 'wind': 3.0, 'water': 1.5, 'vegetation': 1.0}
        profile = "SEVERE STRUCTURAL"
    elif wind_pts > 3.5:
        weights = {'wind': 4.0, 'structural': 3.0, 'vegetation': 2.0, 'water': 1.0}
        profile = "WIND DAMAGE"
    else:
        weights = {'structural': 3.0, 'vegetation': 2.5, 'wind': 2.5, 'water': 2.0}
        profile = "MODERATE MIXED"

    adaptive_score = (vegetation_pts * weights['vegetation'] +
                     structural_pts * weights['structural'] +
                     wind_pts * weights['wind'] +
                     water_pts * weights['water']) / 10.0
    adaptive_score = (adaptive_score / 10) * 100
    print(f"3. Adaptive Profile: {adaptive_score:.1f}/100 ({profile})")

    # Scheme 4: Equal Weight Composite
    # Simple average of all indicators
    equal_score = (structural_pts + water_pts + wind_pts + vegetation_pts) / 4.0
    equal_score = (equal_score / 10) * 100
    print(f"4. Equal Weight Composite: {equal_score:.1f}/100")

    # Scheme 5: Severity Amplified
    # Exponentially weights higher severity
    def severity_amplify(pts):
        if pts < 2: return pts * 0.8
        elif pts < 5: return pts * 1.2
        elif pts < 8: return pts * 1.8
        else: return pts * 2.5

    severity_score = (severity_amplify(structural_pts) + severity_amplify(water_pts) +
                     severity_amplify(wind_pts) + severity_amplify(vegetation_pts)) / 4.0
    severity_score = min((severity_score / 10) * 100, 100)
    print(f"5. Severity Amplified: {severity_score:.1f}/100")

    print(f"{'='*70}\n")

    return {
        'urban': urban_score,
        'environmental': environmental_score,
        'adaptive': adaptive_score,
        'equal': equal_score,
        'severity': severity_score,
        'profile': profile,
        'total_points': structural_pts + water_pts + wind_pts + vegetation_pts
    }

# Calculate weighted scores
weighted_results = {}

for region, results in all_results.items():
    weighted_results[region] = calculate_weighted_scores(results, region)

# Display rankings
print("\n" + "="*70)
print("COMPARATIVE RANKINGS BY WEIGHTING SCHEME")
print("="*70)

schemes = {
    'urban': 'Urban/Infrastructure Focus',
    'environmental': 'Environmental/Total (RECOMMENDED)',
    'adaptive': 'Adaptive Profile-Based',
    'equal': 'Equal Weight Composite',
    'severity': 'Severity Amplified'
}

for scheme_key, scheme_name in schemes.items():
    print(f"\n{scheme_name}:")
    print("-" * 70)

    rankings = sorted(
        [(region, scores[scheme_key]) for region, scores in weighted_results.items()],
        key=lambda x: x[1], reverse=True
    )

    for i, (region, score) in enumerate(rankings, 1):
        profile = weighted_results[region].get('profile', '')
        total_pts = weighted_results[region]['total_points']
        profile_str = f" ({profile})" if scheme_key == 'adaptive' else ""
        print(f"  {i}. {region:15s}: {score:5.1f}/100 [{total_pts:.2f}/40 pts]{profile_str}")

# Recommended score with detailed breakdown
print("\n" + "="*70)
print("RECOMMENDED: ENVIRONMENTAL/TOTAL DEVASTATION SCORE")
print("="*70)
print("Best captures landscape-level destruction for Cat 5 hurricanes\n")

env_rankings = sorted(
    [(region, weighted_results[region]['environmental'],
      all_results[region]['structural_points'],
      all_results[region]['water_points'],
      all_results[region]['wind_points'],
      all_results[region]['vegetation_points'],
      all_results[region]['damaged_building_count'])
     for region in all_results.keys()],
    key=lambda x: x[1], reverse=True
)

for i, (region, score, struct_pts, water_pts, wind_pts, veg_pts, bldg_count) in enumerate(env_rankings, 1):
    print(f"{i}. {region:15s}: {score:.1f}/100")
    print(f"   └─ Buildings: {bldg_count:,} damaged | Water: {water_pts:.2f}/10 | "
          f"Wind: {wind_pts:.2f}/10 | Veg: {veg_pts:.2f}/10\n")

# Point breakdown comparison
print("\n" + "="*70)
print("POINT SCORE BREAKDOWN")
print("="*70)
print("Showing raw point scores for direct comparison\n")

point_rankings = sorted(
    [(region, all_results[region]['total_points'],
      all_results[region]['structural_points'],
      all_results[region]['water_points'],
      all_results[region]['wind_points'],
      all_results[region]['vegetation_points'],
      all_results[region]['building_damage_pct'],
      all_results[region]['damaged_building_count'])
     for region in all_results.keys()],
    key=lambda x: x[1], reverse=True
)

for i, (region, total, struct, water, wind, veg, bldg_pct, bldg_count) in enumerate(point_rankings, 1):
    print(f"{i}. {region:15s}: {total:.2f}/40 total points")
    print(f"   Structural (Buildings):  {struct:.2f}/10 ({bldg_count:,} buildings, {bldg_pct:.1f}%)")
    print(f"   Water Damage:            {water:.2f}/10")
    print(f"   Wind Damage:             {wind:.2f}/10")
    print(f"   Vegetation Loss:         {veg:.2f}/10")
    print()


WEIGHTED ANALYSIS: Kingston
Individual Scores (out of 10):
  Structural Collapse:  5.73/10
  Water Damage:         3.95/10
  Wind Damage:          4.68/10
  Vegetation Loss:      1.58/10

1. Urban/Infrastructure Focus: 46.4/100
2. Environmental/Total: 36.7/100
3. Adaptive Profile: 47.3/100 (SEVERE STRUCTURAL)
4. Equal Weight Composite: 39.9/100
5. Severity Amplified: 54.8/100


WEIGHTED ANALYSIS: Ocho Rios
Individual Scores (out of 10):
  Structural Collapse:  2.52/10
  Water Damage:         3.17/10
  Wind Damage:          2.65/10
  Vegetation Loss:      3.40/10

1. Urban/Infrastructure Focus: 27.8/100
2. Environmental/Total: 30.1/100
3. Adaptive Profile: 29.0/100 (MODERATE MIXED)
4. Equal Weight Composite: 29.4/100
5. Severity Amplified: 35.2/100


WEIGHTED ANALYSIS: Negril
Individual Scores (out of 10):
  Structural Collapse:  4.40/10
  Water Damage:         2.66/10
  Wind Damage:          4.04/10
  Vegetation Loss:      3.11/10

1. Urban/Infrastructure Focus: 38.1/100
2. Environmen

## Section 7: Results Analysis and Limitations

### Critical Issue: Rankings May Not Reflect Ground Truth

**News reports indicated:**
- Montego Bay & Falmouth experienced severe damage
- Other areas sustained varying degrees of impact

**Actual Results:**
1. Kingston: 15.94/40 (39.9%)
2. Falmouth: 15.80/40 (39.5%)
3. Negril: 14.21/40 (35.5%)
4. Montego Bay: 14.18/40 (35.5%)
5. Ocho Rios: 11.74/40 (29.4%)

**Observation**: Montego Bay and Falmouth—reported as severely damaged—rank 4th and 2nd respectively, while Kingston ranks first. This raises questions about whether the scoring methodology accurately captures damage severity.

---

### Why the Rankings May Be Problematic

#### 1. **Building Count Dominates Structural Score**

Kingston has **61,966 buildings** vs Falmouth's **817 buildings**:
- Kingston: 57.3% damaged = 5.73 points
- Falmouth: 38.8% damaged = 3.88 points

**Issue**: Kingston's massive building inventory inflates its structural score even if Falmouth experienced more severe per-building damage. The scoring system rewards urban density rather than damage intensity.

**Better approach**: Should weight by *severity of damage* (coherence values) not just *count of damaged buildings*.

#### 2. **Equal Weighting Inappropriate for Different Damage Types**

Each indicator gets 0-10 points with no contextual weighting:
- Structural damage in Kingston (urban center) ≠ vegetation loss in Falmouth (resort town)
- A 5.73 point structural score doesn't necessarily indicate worse conditions than a 5.40 vegetation score

**Issue**: Comparing 61,966 damaged buildings to vegetation loss across a small resort area is meaningless without context-appropriate weights.

#### 3. **Polygon Definition Bias**

Kingston's polygon (133.5 km²) includes extensive urban area → more buildings → higher structural points

Falmouth's polygon (13.8 km²) is smaller and less densely developed → fewer buildings → lower structural points despite potentially higher damage intensity

**Issue**: Larger urban areas automatically score higher in structural metrics regardless of actual damage severity.

#### 4. **Missing Damage Intensity Metrics**

Current scoring only considers:
- **Extent**: % of area/buildings affected
- **Binary thresholds**: Is it damaged (yes/no)?

**Missing**:
- **Severity**: *How badly* is it damaged?
- **Building damage**: coherence 0.69 (minor) vs 0.2 (destroyed) both count as "damaged"
- **Vegetation loss**: NDVI -0.16 (mild) vs -0.6 (catastrophic) both count as "damaged"

---

### What the Data Actually Shows

#### **Kingston (15.94 points) - Ranked 1st**

**Strengths**: High structural damage count (61,966 buildings)  
**Characteristics**: Large urban area with moderate damage across many buildings  
**Why it ranks high**: Building count dominates score

| Metric | Score | Characteristics |
|--------|-------|-----------------|
| Structural | 5.73/10 | 57.3% of 108,000+ buildings = widespread moderate damage |
| Water | 3.95/10 | Moderate flooding signature |
| Wind | 4.68/10 | Moderate debris signature |
| Vegetation | 1.58/10 | Minimal (urban area, less vegetation present) |

#### **Falmouth (15.80 points) - Ranked 2nd**

**Strengths**: Highest vegetation loss (5.40 points)  
**News reports**: Severe damage  
**Why it may be underestimated**: Small building count (817) limits structural score

| Metric | Score | Characteristics |
|--------|-------|-----------------|
| Structural | 3.88/10 | 38.8% of 817 buildings = 317 buildings damaged |
| Water | 2.58/10 | Moderate flooding signature |
| Wind | 3.93/10 | Significant debris signature |
| Vegetation | **5.40/10** | **54% vegetation destroyed** |

**Key observation**: Vegetation loss (54%) may better represent total devastation than building count alone.

#### **Montego Bay (14.18 points) - Ranked 4th**

**Strengths**: High vegetation loss (4.98 points), larger building inventory (10,072)  
**News reports**: Severe damage  
**Why it may be underestimated**: Only 30.6% of buildings flagged (threshold too conservative?)

| Metric | Score | Characteristics |
|--------|-------|-----------------|
| Structural | 3.06/10 | 30.6% of 10,072 = 3,082 buildings damaged |
| Water | 2.78/10 | Moderate flooding signature |
| Wind | 3.37/10 | Moderate wind damage signature |
| Vegetation | **4.98/10** | **50% vegetation lost** - indicates severe impacts |

#### **Negril (14.21 points) - Ranked 3rd**

| Metric | Score | Characteristics |
|--------|-------|-----------------|
| Structural | 4.40/10 | 1,299 buildings, 44% damaged |
| Water | 2.66/10 | Moderate |
| Wind | 4.04/10 | Moderate |
| Vegetation | 3.11/10 | Moderate vegetation loss |

#### **Ocho Rios (11.74 points) - Ranked 5th**

Lowest damage scores across all metrics.

---

### Understanding What Satellite Data Can and Cannot Detect

#### **SAR Coherence Limitations**

**What It Detects Well:**
- Complete building collapse
- Major structural deformation
- Debris field accumulation

**What It Misses:**
- **Flooding** - water remains smooth → high coherence
- **Roof damage without collapse** - building still standing
- **Interior damage** - radar can't penetrate intact roofs
- **Non-structural wind damage** - broken windows, siding removed

**Critical implication**: The 0.7 coherence threshold may still be too conservative. Buildings with severe roof damage (coherence 0.65-0.69) are being missed.

#### **Why Reported Severe Damage ≠ Low Coherence Scores**

Montego Bay and Falmouth reported as severely damaged but show moderate structural scores:

**Possible explanations**:
1. Buildings severely damaged but still standing (coherence 0.65-0.75)
2. Flooding and wind destroyed interiors without collapsing walls
3. Threshold (0.7) misses repairable-but-uninhabitable buildings
4. Debris removal between hurricane and post-event imaging
5. Small building counts (Falmouth) or lower detection rates (Montego Bay) limit structural scores

**Notable**: Both areas show ~50% vegetation loss, suggesting severe environmental impacts.

---

### Fundamental Scoring Problems

#### **Problem 1: Building Count vs Building Severity**

**Current**: 57.3% of 61,966 buildings (Kingston) = 5.73 points  
**Current**: 38.8% of 817 buildings (Falmouth) = 3.88 points

**Issue**: This doesn't account for damage severity
- If Falmouth's 317 damaged buildings have mean coherence 0.3 (destroyed)
- And Kingston's 35,524 damaged buildings have mean coherence 0.65 (repairable)
- Falmouth experienced worse damage despite lower count

#### **Problem 2: No Cross-Indicator Validation**

**Kingston**: High structural (5.73) + low vegetation (1.58) = urban area, moderate damage  
**Falmouth**: Low structural (3.88) + high vegetation (5.40) = severe damage in small area

**These tell different stories** but current system treats them as directly comparable.

#### **Problem 3: Polygon Area Normalization Missing**

Damage per km²:
- Kingston: 15.94 points / 133.5 km² = **0.119 points/km²**
- Falmouth: 15.80 points / 13.8 km² = **1.145 points/km²**

**Falmouth has 9.6× higher damage density** but ranks second due to absolute scoring.

---

### What Should Be Done Differently

#### **1. Severity-Weighted Structural Score**

Instead of: `(damaged_buildings / total_buildings) × 10`

Use: `mean_coherence_decrease_in_damaged_buildings × (damaged_buildings / total_buildings) × 10`

This weights by *how badly* buildings are damaged, not just *how many*.

#### **2. Context-Appropriate Weighting**

- **Urban areas** (Kingston): Structural damage × 3.0, Vegetation × 1.0
- **Resort/coastal** (Falmouth, Negril): Vegetation × 3.0, Structural × 1.5, Water × 2.0
- **Mixed** (Montego Bay): Balanced weights

#### **3. Normalize by Area**

Report both:
- **Absolute score**: Total damage points (current method)
- **Damage density**: Points per km² (reveals intensity)

#### **4. Multi-Tier Structural Classification**

Instead of binary "damaged/not damaged":
- **Destroyed**: coherence < 0.4 (4 points per building)
- **Severe**: coherence 0.4-0.6 (3 points per building)
- **Moderate**: coherence 0.6-0.7 (2 points per building)
- **Minor**: coherence 0.7-0.8 (1 point per building)

This captures damage *intensity* not just presence.

#### **5. Vegetation Loss as Damage Indicator**

For Category 5 hurricanes:
- When vegetation loss >50%, indicates widespread severe impacts
- Weight vegetation more heavily in environmental/total devastation scenarios
- Recognize that destroyed vegetation suggests severe impacts to all infrastructure

**Observation**: Falmouth (54% veg loss) and Montego Bay (50% veg loss)—both reported as severely damaged—show the highest vegetation destruction.

---

### Alternative Interpretation: Vegetation as Primary Indicator

**Ranking by vegetation loss:**

1. **Falmouth**: 54.0% vegetation loss
2. **Montego Bay**: 49.8% vegetation loss
3. **Ocho Rios**: 34.0% vegetation loss
4. **Negril**: 31.1% vegetation loss
5. **Kingston**: 15.8% vegetation loss

**Observation**: This ordering places the two cities reported as severely damaged (Montego Bay and Falmouth) at the top, suggesting vegetation loss may be a better proxy for total devastation in this context.

---

### Key Lessons

1. **Building count ≠ damage severity** - Kingston's high score may be an artifact of urban density
2. **Thresholds need validation** - 0.7 coherence may miss severe but non-collapsed damage
3. **Single metric insufficient** - Structural scores alone may underestimate total devastation
4. **Vegetation loss is informative** - 50%+ vegetation destruction correlates with reported severe damage
5. **Context-specific weighting essential** - Urban vs coastal areas need different scoring approaches
6. **Polygon definition critical** - Size and land use type dramatically affect results
7. **Ground truth validation necessary** - Without field data, threshold calibration remains uncertain

---

### Honest Assessment

**What worked**:
- ✅ Multi-indicator approach captured different damage types
- ✅ Vegetation loss appears to correlate with reported severe damage
- ✅ Methodology was systematic and reproducible

**What needs improvement**:
- ⚠️ Scoring system may favor large urban areas regardless of severity
- ⚠️ Equal weighting may be inappropriate for different contexts
- ⚠️ Thresholds not validated against ground truth data
- ⚠️ Building count may dominate structural score inappropriately
- ⚠️ Rankings may not reflect actual damage severity

**Bottom line**: The satellite data and detection methods appear sound, but the **scoring and aggregation methodology requires validation against ground truth** to determine if it produces meaningful damage rankings.

## Section 8: Interactive Map Visualization

Create an interactive map showing all damage assessment results.

### Layer Types

**Coherence** (Red = damaged, Green = stable): Shows surface change  
**Backscatter Change** (Blue = flooding, Red = debris): Radar signal intensity  
**NDVI Change** (Brown = loss, Green = gain): Vegetation health  
**Binary Damage** (Red): Areas flagged as damaged

### Map Usage Tips

- **Layer Control**: Click the layers icon (top right) to toggle visibility
- **Compare Regions**: Turn on same layer for multiple regions
- **Zoom**: Focus on specific areas to see 10m resolution detail
- **Notice Patterns**: Falmouth/Montego Bay show massive vegetation loss

In [ ]:
# ============================================================================
# SECTION 8: INTERACTIVE MAP VISUALIZATION
# ============================================================================

# Visualization parameters
coherence_vis = {"min": 0, "max": 1, "palette": ["FF0000", "FFFF00", "00FF00"]}
backscatter_vis = {"min": -5, "max": 5, "palette": ["0000FF", "FFFFFF", "FF0000"]}
ndvi_vis = {"min": -0.5, "max": 0.2, "palette": ["8B0000", "FF0000", "FFA500", "FFFF00", "90EE90", "00FF00"]}

# NEW: Continuous damage visualizations (gradient from no damage to severe)
water_vis = {"min": 0, "max": 8, "palette": ["FFFFFF", "ADD8E6", "4682B4", "000080", "00008B"]}  # White to dark blue
wind_vis = {"min": 0, "max": 8, "palette": ["FFFFFF", "FFD700", "FF8C00", "FF4500", "8B0000"]}   # White to dark red
vegetation_vis = {"min": 0, "max": 0.6, "palette": ["FFFFFF", "90EE90", "32CD32", "228B22", "006400"]}  # White to dark green
combined_vis = {"min": 0, "max": 1, "palette": ["FFFFFF", "FFFF00", "FFA500", "FF4500", "8B0000"]}  # White to dark red

# Initialize map
Map = geemap.Map(basemap="SATELLITE", center=[18.2, -77.5], zoom=9)

print("Creating consolidated layers for all regions...")

# Combine all regions for each indicator type into single layers

# 1. Coherence (all regions combined) - CONTINUOUS
coherence_all = ee.ImageCollection([results['coherence'] for results in all_results.values()]).mosaic()
Map.addLayer(coherence_all, coherence_vis, "1. Coherence (Continuous)", False)

# 2. Backscatter Change (all regions combined) - CONTINUOUS
backscatter_all = ee.ImageCollection([results['backscatter_change'] for results in all_results.values()]).mosaic()
Map.addLayer(backscatter_all, backscatter_vis, "2. Backscatter Change (Continuous)", False)

# 3. NDVI Change (all regions combined) - CONTINUOUS
ndvi_all = ee.ImageCollection([results['ndvi_change'] for results in all_results.values()]).mosaic()
Map.addLayer(ndvi_all, ndvi_vis, "3. NDVI Change (Continuous)", False)

# 4. Water Damage (all regions combined) - CONTINUOUS
water_continuous_all = ee.ImageCollection([results['water_damage_continuous'] for results in all_results.values()]).mosaic()
Map.addLayer(water_continuous_all, water_vis, "4. Water Damage (Continuous)", True)

# 5. Wind Damage (all regions combined) - CONTINUOUS
wind_continuous_all = ee.ImageCollection([results['wind_damage_continuous'] for results in all_results.values()]).mosaic()
Map.addLayer(wind_continuous_all, wind_vis, "5. Wind Damage (Continuous)", True)

# 6. Vegetation Loss (all regions combined) - CONTINUOUS
veg_continuous_all = ee.ImageCollection([results['vegetation_damage_continuous'] for results in all_results.values()]).mosaic()
Map.addLayer(veg_continuous_all, vegetation_vis, "6. Vegetation Loss (Continuous)", True)

# 7. Combined Damage (all regions combined) - CONTINUOUS
combined_continuous_all = ee.ImageCollection([results['combined_damage_continuous'] for results in all_results.values()]).mosaic()
Map.addLayer(combined_continuous_all, combined_vis, "7. Combined Damage (Continuous)", True)

# 8. Structural Damage - Binary (for reference)
structural_all = ee.ImageCollection([results['structural_damage'] for results in all_results.values()]).mosaic()
Map.addLayer(structural_all, {"palette": ["FF0000"]}, "8. Structural Damage (Binary)", False)

# 9. Region boundaries for reference
for name, aoi in aois.items():
    Map.addLayer(aoi, {'color': 'yellow'}, f"{name} Boundary", False)

print("\n" + "="*70)
print("INTERACTIVE MAP READY")
print("="*70)
print("Total layers: 7 continuous + 1 binary + 5 boundaries = 13 layers")
print("\nLayer Guide (CONTINUOUS = pixel-by-pixel severity):")
print("  • Coherence: Red=damaged, Yellow=moderate, Green=stable")
print("  • Backscatter Change: Blue=flooding, White=no change, Red=debris")
print("  • NDVI Change: Brown=vegetation loss, Green=vegetation gain")
print("  • Water Damage: White→Dark Blue (0 to 8 dB decrease)")
print("  • Wind Damage: White→Dark Red (0 to 8 dB increase)")
print("  • Vegetation Loss: White→Dark Green (0 to 0.6 NDVI loss)")
print("  • Combined Damage: White→Dark Red (composite severity 0-1)")
print("  • Structural Damage: Binary red mask (coherence < 0.5)")
print("\nContinuous layers show GRADIENTS of damage severity!")
print("="*70)

Map

Creating consolidated layers for all regions...

INTERACTIVE MAP READY
Total layers: 7 continuous + 1 binary + 5 boundaries = 13 layers

Layer Guide (CONTINUOUS = pixel-by-pixel severity):
  • Coherence: Red=damaged, Yellow=moderate, Green=stable
  • Backscatter Change: Blue=flooding, White=no change, Red=debris
  • NDVI Change: Brown=vegetation loss, Green=vegetation gain
  • Water Damage: White→Dark Blue (0 to 8 dB decrease)
  • Wind Damage: White→Dark Red (0 to 8 dB increase)
  • Vegetation Loss: White→Dark Green (0 to 0.6 NDVI loss)
  • Combined Damage: White→Dark Red (composite severity 0-1)
  • Structural Damage: Binary red mask (coherence < 0.5)

Continuous layers show GRADIENTS of damage severity!


Map(center=[18.2, -77.5], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI…

## Section 9: Final Summary and Conclusions

### Final Damage Rankings

**Point-Based Scoring** (0-40 points total):
- Structural (Buildings): 0-10 points based on % of buildings damaged
- Water Damage: 0-10 points (extent 0-5 + intensity 0-5)
- Wind Damage: 0-10 points (extent 0-5 + intensity 0-5)
- Vegetation Loss: 0-10 points based on % of area with NDVI decrease > 0.15

**Results:**

1. **Kingston**: 15.94/40 (39.9%) - 61,966 buildings damaged
2. **Falmouth**: 15.80/40 (39.5%) - 817 buildings damaged, 54% vegetation loss
3. **Negril**: 14.21/40 (35.5%) - 1,299 buildings damaged
4. **Montego Bay**: 14.18/40 (35.5%) - 10,072 buildings damaged, 50% vegetation loss
5. **Ocho Rios**: 11.74/40 (29.4%) - 1,996 buildings damaged

### Important Caveats

**These rankings should be interpreted cautiously:**
- Kingston's top ranking may be an artifact of large urban area and high building count
- Montego Bay and Falmouth—reported as severely damaged—rank 4th and 2nd
- Vegetation loss (50%+ for Montego Bay and Falmouth) may better indicate total devastation
- No context-specific weighting applied (urban vs coastal areas treated equally)
- Damage density not considered (points per km²)

**Alternative interpretation:** Ranking by vegetation loss places Falmouth (54%) and Montego Bay (50%) at the top, aligning better with news reports of severe damage.

In [ ]:
# ============================================================================
# SECTION 9: FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("FINAL DAMAGE RANKINGS - Hurricane Melissa")
print("="*80)

final_rankings = sorted(
    [(region,
      all_results[region]['total_points'],
      all_results[region]['normalized_score'],
      all_results[region]['building_damage_pct'],
      all_results[region]['damaged_building_count'],
      all_results[region]['water_pct'],
      all_results[region]['wind_pct'],
      all_results[region]['vegetation_pct'])
     for region in all_results.keys()],
    key=lambda x: x[1], reverse=True
)

print("\nRanked by Total Damage Score (0-40 points):\n")

for i, (region, total_pts, norm_score, bldg_pct, bldg_count, water, wind, veg) in enumerate(final_rankings, 1):
    print(f"{i}. {region:15s}: {total_pts:5.2f}/40 points ({norm_score:5.1f}%)")
    print(f"   Buildings: {bldg_count:,} damaged ({bldg_pct:5.1f}%)")
    print(f"   Water: {water:5.1f}% | Wind: {wind:5.1f}% | Vegetation: {veg:5.1f}%")

    if veg > 50:
        print(f"   → Catastrophic vegetation loss ({veg:.1f}%)")
    elif total_pts > 15:
        print(f"   → Severe multi-indicator damage")
    elif veg > 40:
        print(f"   → Significant environmental impact")

    print()

print("="*80)
print("IMPORTANT NOTES:")
print("="*80)
print("• Rankings based on equal weighting across all indicators")
print("• Kingston's high score may reflect urban density rather than severity")
print("• Montego Bay and Falmouth show highest vegetation loss (50%+)")
print("• Vegetation loss may better indicate total devastation for coastal areas")
print("• No ground truth validation - rankings should be interpreted cautiously")
print("="*80)
print("\nAnalysis complete.")
print("="*80)


FINAL DAMAGE RANKINGS - Hurricane Melissa

Ranked by Total Damage Score (0-40 points):

1. Kingston       : 15.94/40 points ( 39.9%)
   Buildings: 61,966 damaged ( 57.3%)
   Water:  27.5% | Wind:  38.9% | Vegetation:  15.8%
   → Severe multi-indicator damage

2. Falmouth       : 15.80/40 points ( 39.5%)
   Buildings: 817 damaged ( 38.8%)
   Water:  15.0% | Wind:  36.9% | Vegetation:  54.0%
   → Catastrophic vegetation loss (54.0%)

3. Negril         : 14.21/40 points ( 35.5%)
   Buildings: 1,299 damaged ( 44.0%)
   Water:  15.3% | Wind:  38.4% | Vegetation:  31.1%

4. Montego Bay    : 14.18/40 points ( 35.5%)
   Buildings: 10,072 damaged ( 30.6%)
   Water:  18.8% | Wind:  29.0% | Vegetation:  49.8%
   → Significant environmental impact

5. Ocho Rios      : 11.74/40 points ( 29.4%)
   Buildings: 1,996 damaged ( 25.2%)
   Water:  26.6% | Wind:  18.3% | Vegetation:  34.0%

IMPORTANT NOTES:
• Rankings based on equal weighting across all indicators
• Kingston's high score may reflect urban

### Key Findings

1. **Ranking Discrepancy**: Point-based scoring places Kingston first (15.94/40) despite Montego Bay and Falmouth being reported as most severely damaged
2. **Vegetation Loss Correlation**: Falmouth (54%) and Montego Bay (50%) show highest vegetation destruction, aligning with news reports of severe damage
3. **Building Count Bias**: Kingston's 61,966 damaged buildings dominate structural score despite potentially lower per-building severity
4. **Multi-Indicator Value**: Different damage signatures captured—structural, water, wind, and vegetation—provide complementary information
5. **Threshold Uncertainty**: Current thresholds (coherence <0.7, backscatter ±2dB, NDVI <-0.15) lack ground truth validation

### Limitations and Challenges

**Scoring Methodology Issues:**
- Equal weighting may be inappropriate for urban vs coastal contexts
- Building count favors large urban areas regardless of damage intensity
- No severity weighting—coherence 0.3 (destroyed) and 0.69 (minor) both count as "damaged"
- Polygon size affects results (Kingston 133.5 km² vs Falmouth 13.8 km²)

**Detection Limitations:**
- SAR coherence misses flooding (water remains smooth)
- Roof damage without collapse may go undetected
- Three of four indicators rely on single sensor (Sentinel-1)
- NDVI analysis cloud-dependent (common issue post-hurricane)

**Validation Gap:**
- No ground truth damage assessments available for calibration
- Thresholds determined through iterative AI-assisted reasoning, not field data
- Cannot confirm if 0.7 coherence threshold appropriately captures damage

### Alternative Interpretation

**Ranking by vegetation loss (may better indicate total devastation):**

1. Falmouth: 54.0% vegetation loss
2. Montego Bay: 49.8% vegetation loss
3. Ocho Rios: 34.0% vegetation loss
4. Negril: 31.1% vegetation loss
5. Kingston: 15.8% vegetation loss

This ordering places the two cities reported as severely damaged at the top, suggesting vegetation destruction may serve as a proxy for total devastation in coastal hurricane contexts.

### What Worked

✅ Multi-indicator approach captured different damage types  
✅ Automated image pair selection ensured adequate coverage and temporal correlation  
✅ Per-building assessment more meaningful than pixel-based metrics  
✅ Vegetation loss appears to correlate with reported damage severity  
✅ Methodology systematic, reproducible, and well-documented  

### What Needs Improvement

⚠️ Scoring system requires validation against ground truth  
⚠️ Context-specific weighting needed (urban vs coastal areas)  
⚠️ Damage intensity not captured—only extent and binary thresholds  
⚠️ Single-sensor dependency (75% of indicators from Sentinel-1)  
⚠️ Polygon definition significantly affects results  

### Recommendations for Future Work

**Immediate Improvements:**
- Implement severity-weighted scoring (mean coherence in damaged buildings)
- Normalize by area (damage density per km²)
- Develop context-appropriate weighting schemes
- Add multi-tier damage classification (destroyed/severe/moderate/minor)

**Long-term Validation:**
- Calibrate thresholds using historical hurricane ground truth data
- Validate against field damage assessments
- Compare results with insurance claims data
- Test methodology across multiple hurricane events

**Methodological Extensions:**
- Incorporate additional data sources (aerial imagery, social media, mobile data)
- Explore machine learning approaches for damage classification
- Develop automated threshold optimization
- Investigate change detection methods beyond coherence

### Data Summary

- **Sentinel-1 SAR**: 10m resolution, C-band VV polarization, IW mode
- **Sentinel-2 Optical**: 10m resolution, multispectral (NIR + Red for NDVI)
- **Coverage**: Five urban areas across Jamaica
- **Buildings Analyzed**: 182,150 total (61,966 Kingston, 10,072 Montego Bay, 2,107 Ocho R